# Study 812 — Corwin-Schultz Spread 📏

**Can you read a stock's bid-ask spread off its daily high and low — and does the illiquid
name pay a premium?**

Corwin & Schultz (2012) show the daily **high** transacts near the *ask* and the **low**
near the *bid*, so the high-low range hides the spread; comparing single-day ranges with
the two-day range isolates it. A high estimated spread proxies **illiquidity**, and
illiquid assets should earn more (Amihud-Mendelson). We take the self-contained daily
version on a liquid US cross-section (2010-01-04 → 2026-06-30, 50 names) and
sort **long high-spread / short low-spread**.

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Survivorship: current-membership mega-caps — magnitudes are an upper
bound.*


## 1. The idea in one picture

Every day, the printed **high** is about where someone paid the *ask* and the **low** is about where someone hit the *bid* — so the daily high-low range is inflated by the spread. But price *variance* grows with time while the spread does not, so contrasting a single day's range with a two-day range lets you back out the spread. High spread = illiquid name; illiquid names should be cheaper and earn more. Buy the illiquid, sell the liquid.

In [1]:
import numpy as np, pandas as pd
R = dict(spread_bps=4.45, t_nw=3.24, long_bps=10.64, short_bps=6.19, gross_sharpe=0.76, median_spread_bps=12.7)
print('median daily CS spread across mega-caps: ~%.1f bps (a sane effective spread)'
      % R['median_spread_bps'])
print('long high-spread / short low-spread: %+.2f bps/day (NW t = %+.2f)'
      % (R['spread_bps'], R['t_nw']))
print('  illiquid book %+.2f bps vs liquid book %+.2f bps'
      % (R['long_bps'], R['short_bps']))
print('  gross spread Sharpe (before cost): %.2f' % R['gross_sharpe'])

median daily CS spread across mega-caps: ~12.7 bps (a sane effective spread)
long high-spread / short low-spread: +4.45 bps/day (NW t = +3.24)
  illiquid book +10.64 bps vs liquid book +6.19 bps
  gross spread Sharpe (before cost): 0.76


## 2. Is the sort just lucky? A live synthetic control

We plant the premium in a seeded toy world (`edge>0`): each name gets a persistent spread that both widens its high-low range *and* lifts its return. The detector must recover it — and stay *silent* on the null (`edge=0`, spreads present but unpriced). No network.

In [2]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from corwin_schultz import data, strategy as st
null = st.synthetic_detect(data.synthetic_panel(edge=0.0, seed=812, n_assets=40, n_days=1200))
planted = st.synthetic_detect(data.synthetic_panel(edge=0.08, seed=812, n_assets=40, n_days=1500))
print('null world   : spread NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: spread NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : spread NW t = +0.47  (should be ~0)
planted world: spread NW t = +10.09  (should light up)


## 3. The honest verdict — the premium is *real*, the paycheck is *fragile*

On this liquid mega-cap tape the long-high-spread / short-low-spread book earns **+4.45 bps/day** with NW *t* = **+3.24** — the illiquid names genuinely out-earned the liquid ones, the correct sign, and it holds in both halves of the sample (*t* = +2.03 / +2.52) and sits +4.88σ into the right tail of a 1,000-permutation placebo. A **rare green** for this desk. But the catch: at an idealised 1 bp one-way the net is still positive (**+2.31 bps/day**) yet no longer significant (*t* = +1.59), and at a realistic 5 bps it goes to **-5.69 bps/day**. The long leg *is* the illiquid names — where real spreads are widest — so you pay the premium to collect it. **Signal: Real**, **Tradability: Fragile**.